# 04 — Análise, controle de qualidade e produtos

**Entrada:** nuvem registrada e pontos de verificação independentes. **Saída:** resultados de acurácia, produtos derivados e relatório técnico.

> Acurácia do sensor, precisão do registro e acurácia do produto são conceitos diferentes. O aceite deve usar observações independentes.

## Objetivos de aprendizagem

1. calcular resíduos, viés, RMSE planimétrico, altimétrico e 3D;
2. avaliar distribuição espacial, completude e consistência do registro;
3. extrair e classificar feições com validação;
4. gerar produtos 3D adequados ao uso declarado;
5. documentar limitações, parâmetros e rastreabilidade;
6. emitir recomendação objetiva de aceite, correção ou nova aquisição.

## 1. Controle posicional

Use pontos de verificação não empregados no ajuste, bem distribuídos em planta e altura. Documente método de identificação na nuvem, incerteza da referência, CRS, época e referencial vertical. O exemplo abaixo é sintético; substitua pelos dados levantados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

verificacao = pd.DataFrame({
    'id': ['PV-01','PV-02','PV-03','PV-04','PV-05','PV-06'],
    'E_ref': [500000,500020,500040,500010,500030,500050],
    'N_ref': [7450000,7450005,7450010,7450030,7450035,7450040],
    'H_ref': [10.00,10.08,10.12,12.02,12.10,12.15],
    'E_nuvem': [500000.02,500020.01,500039.97,500010.03,500030.01,500049.96],
    'N_nuvem': [7449999.98,7450005.03,7450010.01,7450029.96,7450035.02,7450040.03],
    'H_nuvem': [10.03,10.06,10.16,12.00,12.13,12.11],
})
for eixo in ['E','N','H']:
    verificacao[f'd{eixo}'] = verificacao[f'{eixo}_nuvem'] - verificacao[f'{eixo}_ref']
verificacao['d2D'] = np.hypot(verificacao.dE, verificacao.dN)
verificacao['d3D'] = np.sqrt(verificacao.dE**2 + verificacao.dN**2 + verificacao.dH**2)
verificacao.round(3)

In [ ]:
def rmse(valores):
    valores = np.asarray(valores, dtype=float)
    return np.sqrt(np.mean(valores**2))

metricas = pd.Series({
    'viés E (m)': verificacao.dE.mean(),
    'viés N (m)': verificacao.dN.mean(),
    'viés H (m)': verificacao.dH.mean(),
    'RMSE 2D (m)': np.sqrt(np.mean(verificacao.dE**2 + verificacao.dN**2)),
    'RMSE H (m)': rmse(verificacao.dH),
    'RMSE 3D (m)': rmse(verificacao.d3D),
    'erro 3D máximo (m)': verificacao.d3D.max(),
})
metricas.round(3).to_frame('resultado')

### 1.1 Interpretação

Compare os valores com o limite contratado e com a incerteza do método de referência. Não conclua apenas pelo RMSE: examine viés, máximos, outliers, quantidade e distribuição dos pontos. Uma amostra pequena não sustenta inferências amplas.

In [ ]:
limite_rmse_3d_m = 0.10  # exemplo; substituir pelo requisito aprovado
resultado_aceite = 'ATENDE' if metricas['RMSE 3D (m)'] <= limite_rmse_3d_m else 'NÃO ATENDE'
print(f'Critério didático: RMSE 3D <= {limite_rmse_3d_m:.2f} m — {resultado_aceite}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.quiver(verificacao.E_ref, verificacao.N_ref, verificacao.dE, verificacao.dN, angles='xy', scale_units='xy', scale=1)
ax.scatter(verificacao.E_ref, verificacao.N_ref, s=15)
ax.set(title='Resíduos planimétricos nos pontos de verificação', xlabel='E (m)', ylabel='N (m)', aspect='equal')
ax.ticklabel_format(style='plain', useOffset=False); ax.grid(alpha=.3); plt.show()

## 2. Completude e consistência geométrica

Além da posição, avalie:

- porcentagem da área obrigatória com cobertura;
- densidade e espaçamento por classe/zona, não só média global;
- lacunas e oclusões em elementos críticos;
- espessura de planos, ruído e superfícies duplicadas;
- resíduos entre passagens e nas regiões de loop;
- coerência de intensidade, cor e tempo, quando usados.

Mapas de densidade, distância nuvem–nuvem/nuvem–malha e cortes transversais ajudam a localizar problemas que uma única métrica esconde.

## 3. Extração e classificação de feições

Uma sequência inicial pode usar altura, normal, verticalidade, rugosidade, retorno/intensidade e cor para separar pavimento, fachadas e vegetação. Regras simples são úteis para protótipos; modelos automáticos precisam de amostra rotulada independente, matriz de confusão, precisão/revocação por classe e inspeção espacial dos erros.

Nunca trate a classificação automática como verdade de campo. Registre versão do modelo, dados de treinamento, classes, limiares e correções manuais.

In [ ]:
matriz = pd.DataFrame(
    [[920, 35, 45], [28, 870, 102], [20, 65, 915]],
    index=pd.Index(['pavimento','fachada','vegetação'], name='referência'),
    columns=pd.Index(['pavimento','fachada','vegetação'], name='predição'),
)
acuracia_global = np.trace(matriz.to_numpy()) / matriz.to_numpy().sum()
recall = np.diag(matriz) / matriz.sum(axis=1)
precisao = np.diag(matriz) / matriz.sum(axis=0)
pd.DataFrame({'precisão': precisao, 'revocação': recall}).round(3).assign(acuracia_global=round(acuracia_global, 3))

## 4. Produtos, visualização e BIM básico

- **Nuvem colorida/classificada:** LAS/LAZ mestre e E57/PLY quando necessário.
- **Malha texturizada:** documente algoritmo, resolução, preenchimento de buracos e decimação.
- **Viewer/flythrough:** produto de comunicação, com versão leve e instruções.
- **IFC básico:** modele objetos e semântica exigidos; não converta milhões de pontos diretamente em entidades BIM. Vincule a nuvem como referência e informe nível de informação/precisão.

Antes da exportação final, valide CRS/unidades, escala, orientação, cores, classes, cabeçalho, abertura no software do cliente e ausência de informações sensíveis.

## 5. Relatório técnico e decisão

O relatório deve conter objetivo, área/data/equipe, equipamento e versões, configuração, apoio e referenciais, trajetórias, processamento, controle de qualidade, resultados, limitações, produtos, licenças e recomendação. A conclusão deve escolher uma ação: **aceitar**, **aceitar com ressalvas**, **reprocessar** ou **readquirir**.

- [ ] critérios de aceite comparados a resultados independentes;
- [ ] gráficos, tabelas e unidades conferidos;
- [ ] parâmetros e fontes rastreáveis;
- [ ] limitações e zonas não cobertas explicitadas;
- [ ] todos os produtos abrem e preservam CRS/metadados.

**Entrega do módulo:** relatório de acurácia, produtos validados e recomendação técnica.

**Próximo módulo:** [05 — Encerramento](05_Encerramento.ipynb).